In [1]:
# Connecting to GitHub

setwd("~/stat_app")

In [2]:
# Installing libraries

install.packages("fixest")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [3]:
# Importing libraries

library(haven)
library(fixest)

In [4]:
# Importing data

wave2 <- read_dta("waveII.dta") # midline (cf. pag. 13)
wave3 <- read_dta("waveIII.dta") # endline

Let's remember our econometric equation:

$Y_{icu} = \alpha + \beta_1 I_c + \beta_2 E_c + \beta_3 (I_c \times E_c) + \beta_4' X_{ic} + \epsilon_{icu}$

where $Y_{icu}$ is the outcome for person $i$ in community $c$ and union $u$, $I_c$ is assignment of community $c$ to the incentive program, $E_c$ is assignment of community $c$ to the empowerment program, and $X_{ic}$ is a vector of individual and community controls measured at baseline for strata, age indicators, household size, the presence of an older unmarried sister in the household, school enrollment, mother’s level of education, and whether the community is accessible via public transport (cf. pag. 16).

In [5]:
# Making the baseline vector, X_ic

controls <- c("older_sister", "bl_still_in_school", "bl_education_mother", "bl_HHsize", "bl_public_transit", "bl_age10", "bl_age11", "bl_age12", "bl_age13",
              "bl_age14", "bl_age15", "bl_age16", "bl_age17", "older_sister_miss", "bl_still_in_school_miss", "bl_education_mother_miss", "bl_HHsize_miss", 
              "bl_public_transit_miss")

In [6]:
# Preparating the regression

eq <- paste("anyemp + anyoil + oil_kk + factor(third) + ", controls, collapse = " + ")

In [7]:
# Sampling at the midline.

dfw2_all <- wave2[wave2$midline == 1 & wave2$washedout == 0 & wave2$before_miss == 0 & wave2$bl_age_reported >= 14 & wave2$bl_age_reported <= 16,]

dfw2_15 <- dfw2_all[dfw2_all$bl_age_reported == 14,]

# Running some regressions

f_ml <- as.formula(paste0("ml_ever_married ~ ", eq, " | unionID"))

regr4 <- feols(f_ml, data = dfw2_all, cluster = ~CLUSTER)
regr5 <- feols(f_ml, data = dfw2_15,  cluster = ~CLUSTER)

NOTE: 140 observations removed because of NA values (LHS: 140).

The variable 'bl_age16' has been removed because of collinearity (see
$collin.var).

NOTE: 52 observations removed because of NA values (LHS: 52).

The variable 'bl_age14' has been removed because of collinearity (see
$collin.var).



In [8]:
# Sampling at the endline

dfw3_all <- wave3[wave3$endline == 1 & wave3$washedout == 0 & wave3$before_miss == 0 & wave3$bl_ever_married == 0 & wave3$bl_age_reported >= 14
                  & wave3$bl_age_reported <= 16,]

dfw3_15 <- dfw3_all[dfw3_all$bl_age_reported == 14, ]

# Running some regressions

f_u18   <- as.formula(paste0("under_18 ~ ", eq, " | unionID"))
f_u16   <- as.formula(paste0("under_16 ~ ", eq, " | unionID"))
f_mar   <- as.formula(paste0("ever_married ~ ", eq, " | unionID"))
f_mage  <- as.formula(paste0("marriage_age ~ ", eq, " | unionID"))
f_b20   <- as.formula(paste0("ever_birth_20 ~ ", eq, " | unionID"))

regr1 <- feols(f_u18,  data = dfw3_all, cluster = ~CLUSTER)
regr2 <- feols(f_u18,  data = dfw3_15,  cluster = ~CLUSTER)
regr3 <- feols(f_u16,  data = dfw3_15,  cluster = ~CLUSTER)
regr6 <- feols(f_mar,  data = dfw3_all, cluster = ~CLUSTER)
regr7 <- feols(f_mar,  data = dfw3_15,  cluster = ~CLUSTER)
regr8 <- feols(f_mage, data = dfw3_all, cluster = ~CLUSTER)
regr9 <- feols(f_mage, data = dfw3_15,  cluster = ~CLUSTER)
regr10 <- feols(f_b20, data = dfw3_all, cluster = ~CLUSTER)
regr11 <- feols(f_b20, data = dfw3_15,  cluster = ~CLUSTER)

NOTE: 27 observations removed because of NA values (LHS: 27).

The variable 'bl_age16' has been removed because of collinearity (see
$collin.var).

NOTE: 10 observations removed because of NA values (LHS: 10).

The variable 'bl_age14' has been removed because of collinearity (see
$collin.var).

NOTE: 10 observations removed because of NA values (LHS: 10).

The variable 'bl_age14' has been removed because of collinearity (see
$collin.var).

NOTE: 14 observations removed because of NA values (LHS: 14).

The variable 'bl_age16' has been removed because of collinearity (see
$collin.var).

NOTE: 7 observations removed because of NA values (LHS: 7).

The variable 'bl_age14' has been removed because of collinearity (see
$collin.var).

NOTE: 2,583 observations removed because of NA values (LHS: 2,583).

The variable 'bl_age16' has been removed because of collinearity (see
$collin.var).

NOTE: 1,098 observations removed because of NA values (LHS: 1,098).

The variable 'bl_age14' has been remove

In [9]:
# Table

etable(regr1, regr2, regr3, regr4, regr5, regr6, regr7, regr8, regr9, regr10, regr11, keep = c("%anyemp","%anyoil","%oil_kk"),
  dict = c(anyemp = "Empowerment", anyoil = "Incentive", oil_kk = "Incen.*Empow."))

,,regr1,regr2,regr3,regr4,regr5,regr6,regr7,regr8,regr9,regr10,regr11
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Dependent Var.:,under_18,under_18,under_16,ml_ever_married,ml_ever_married,ever_married,ever_married,marriage_age,marriage_age,ever_birth_20,ever_birth_20
2,,,,,,,,,,,,
3,Empowerment,-0.0073 (0.0079),-0.0053 (0.0147),0.0061 (0.0092),0.0110 (0.0113),0.0090 (0.0170),0.0057 (0.0076),0.0017 (0.0116),0.0122 (0.0397),0.0026 (0.0648),0.0056 (0.0073),0.0054 (0.0126)
4,Incentive,-0.0488*** (0.0099),-0.0743*** (0.0191),-0.0197. (0.0116),-0.0248. (0.0127),-0.0536** (0.0189),-0.0094 (0.0104),-0.0182 (0.0155),0.2103*** (0.0509),0.3231*** (0.0794),-0.0157. (0.0094),-0.0393* (0.0159)
5,Incen.*Empow.,0.0190 (0.0144),0.0279 (0.0263),-0.0023 (0.0164),-0.0112 (0.0188),0.0072 (0.0259),-0.0022 (0.0139),0.0054 (0.0216),-0.0513 (0.0738),-0.0903 (0.1182),-0.0026 (0.0139),0.0110 (0.0232)
6,Fixed-Effects:,-------------------,-------------------,-----------------,-----------------,------------------,----------------,----------------,------------------,------------------,-----------------,-----------------
7,unionID,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
8,_______________,___________________,___________________,_________________,_________________,__________________,________________,________________,__________________,__________________,_________________,_________________
9,S.E.: Clustered,by: CLUSTER,by: CLUSTER,by: CLUSTER,by: CLUSTER,by: CLUSTER,by: CLUSTER,by: CLUSTER,by: CLUSTER,by: CLUSTER,by: CLUSTER,by: CLUSTER
